In [1]:
import pandas as pd
import geopandas as gpd
import overturemaps 
from shapely import wkb
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_categorical_cmap
import folium
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 500) # Allows to inspect all the columns present in the dataframe

In [ ]:
# Specify bounding box
east, south, west, north = (36.62768265193316, 37.561544985221865, 37.76386851796805, 38.460405921314695)
# north, south, east, west = -32.764304,-32.800422,-61.587818,-61.624380 #armstrong, argentina
bbox = east, south, west, north # Use the 50km rectangle

In [12]:
# Download builidngs from overture
table = overturemaps.record_batch_reader("place", bbox).read_all()
table = table.combine_chunks()
# Convert to dataframe
df = table.to_pandas()

In [16]:
for cat in df.basic_category.unique():
    if 'hospital' in cat:
        print(cat)
    elif 'care' in cat:
        print(cat)
    elif 'clinic' in cat:
        print(cat)

hospital
child_care_or_day_care
healthcare_location
eye_care
clinic_or_treatment_center
orthodontic_care


In [34]:
df['primary_name'] = df['names'].apply(lambda x: x['primary'] if isinstance(x, dict) and 'primary' in x else None)

In [36]:
from shapely.geometry import box
from shapely.geometry import Point

# Extract bbox coordinates and create polygon geometries

# Create a list to store geometries
geometries = []
for idx, row in df.iterrows():
    bbox_dict = row['bbox']
    # Create a Point from one of the external corners of the bbox
    geom = Point(bbox_dict['xmin'], bbox_dict['ymin'])
    geometries.append(geom)
# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometries, crs='EPSG:4326')

In [37]:
# gdf[gdf['basic_category'].isin(['hospital', 'healthcare_location', 'clinic_or_treatment_center'])][['geometry']].explore()

In [45]:
m = folium.Map(location=(38.236801, 36.896292), zoom_start = 15)
gdf[gdf['basic_category'].isin(['hospital', 'healthcare_location', 'clinic_or_treatment_center'])][['primary_name', 'basic_category', 'geometry']].explore(m = m)
m.save('healthcare_locations.html')
m